In [1]:
import cv2
import numpy as np
import subprocess
from PIL import Image, ImageDraw, ImageFont


In [ ]:

# # --- Configuraciones del video ---
# ancho = 720
# alto = 1280
# fps = 30
# duracion = 10  # Duración en segundos
# nombre_archivo = "video_opencv_desde_cero.mp4"

# # --- Inicializar el objeto VideoWriter ---
# fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Códec para .mp4 (puede variar según el sistema)
# video_writer = cv2.VideoWriter(nombre_archivo, fourcc, fps, (ancho, alto))

# # --- Generar los fotogramas y escribir al video ---
# num_frames = int(duracion * fps)

# for i in range(num_frames):
#     # Calcular el color del fondo gradualmente (ejemplo: cambio en el canal azul)
#     blue_channel = int((i / num_frames) * 255)
#     green_channel = 150
#     red_channel = 200
#     color_fondo = (blue_channel, green_channel, red_channel)  # OpenCV usa BGR

#     # Crear un fotograma con el color de fondo
#     frame = np.full((alto, ancho, 3), color_fondo, dtype=np.uint8)

#     # Escribir el fotograma al archivo de video
#     video_writer.write(frame)

# # --- Liberar el objeto VideoWriter ---
# video_writer.release()

# print(f"¡Video '{nombre_archivo}' creado exitosamente!")

¡Video 'video_opencv_desde_cero.mp4' creado exitosamente!


In [2]:
def overlay_logo(frame):
    """Superpone el logo en la esquina inferior derecha."""
    # Cargar logo opcional (si existe)
    try:
        logo = cv2.imread('Carretero.png', cv2.IMREAD_UNCHANGED)  # Para PNGs con transparencia
        if logo is not None:
            # Redimensionar el logo
            scale_factor = 100 / logo.shape[0]  # Alto a 100px
            logo = cv2.resize(logo, (0, 0), fx=scale_factor, fy=scale_factor)
        else:
            logo = None
    
        lh, lw = logo.shape[:2]
        fh, fw = frame.shape[:2]

        # Coordenadas
        x_offset = fw - lw - 20
        y_offset = fh - lh - 20

        # Si el logo tiene canal alpha (transparencia)
        if logo.shape[2] == 4:
            alpha_logo = logo[:, :, 3] / 255.0
            alpha_frame = 1.0 - alpha_logo

            for c in range(0, 3):
                frame[y_offset:y_offset+lh, x_offset:x_offset+lw, c] = (
                    alpha_logo * logo[:, :, c] +
                    alpha_frame * frame[y_offset:y_offset+lh, x_offset:x_offset+lw, c]
                )
        else:
            frame[y_offset:y_offset+lh, x_offset:x_offset+lw] = logo
    except:
        print("error en logo")
    
    return frame

## Frames con Background

In [3]:
def generate_background(t):
    """Genera un fondo hipnótico oscuro y tétrico."""
    w, h = VIDEO_SIZE
    x = np.linspace(0, 4*np.pi, w)
    y = np.linspace(0, 4*np.pi, h)
    X, Y = np.meshgrid(x, y)

    # Generar una onda más caótica
    wave = (np.sin(X * 2 + t*0.5) + np.sin(Y * 2 - t*1.5) + np.sin(X + Y + t)) * 0.33
    norm_wave = ((wave - wave.min()) / (wave.max() - wave.min()) * 255).astype(np.uint8)

    frame = np.zeros((h, w, 3), dtype=np.uint8)

    # Colores más tétricos: mezcla de rojo oscuro y azul profundo
    frame[..., 0] = (norm_wave // 3)  # Azul muy oscuro
    frame[..., 1] = (norm_wave // 8)  # Verde casi ausente
    frame[..., 2] = (norm_wave // 2) + 10  # Rojo oscuro profundo

    return frame

In [ ]:
def generate_background(t):
    """Genera el fondo hipnótico."""
    w, h = VIDEO_SIZE
    x = np.linspace(0, 2*np.pi, w)
    y = np.linspace(0, 2*np.pi, h)
    X, Y = np.meshgrid(x, y)

    wave = (np.sin(X * 2 + t) + np.sin(Y * 3 - t*1.5)) * 0.5
    norm_wave = ((wave - wave.min()) / (wave.max() - wave.min()) * 255).astype(np.uint8)

    frame = np.zeros((h, w, 3), dtype=np.uint8)
    frame[..., 0] = norm_wave
    frame[..., 1] = (norm_wave // 2) + 60
    frame[..., 2] = 255 - norm_wave

    return frame

In [ ]:
def generate_background_dark(t, flash_times=None):
    """
    Genera un fondo oscuro hipnótico.
    Si el tiempo t está cerca de uno de los flash_times, hace un efecto de parpadeo.
    
    flash_times: lista de segundos donde quieres que haya parpadeo.
    """
    w, h = VIDEO_SIZE
    x = np.linspace(0, 4*np.pi, w)
    y = np.linspace(0, 4*np.pi, h)
    X, Y = np.meshgrid(x, y)

    # Onda base oscura
    wave = (np.sin(X * 2 + t*0.5) + np.sin(Y * 2 - t*1.5) + np.sin(X + Y + t)) * 0.33
    norm_wave = ((wave - wave.min()) / (wave.max() - wave.min()) * 255).astype(np.uint8)

    frame = np.zeros((h, w, 3), dtype=np.uint8)
    frame[..., 0] = (norm_wave // 3)    # Azul oscuro
    frame[..., 1] = (norm_wave // 8)    # Verde casi inexistente
    frame[..., 2] = (norm_wave // 2) + 10  # Rojo profundo

    # --- Efecto de parpadeo ---
    if flash_times:
        for flash_t in flash_times:
            if abs(t - flash_t) < 0.15:  # si estamos a ±0.15 segundos del flash
                # Intensificar colores brevemente
                frame = np.clip(frame * 2, 0, 255).astype(np.uint8)
                break  # No seguir revisando otros flashes si ya hay uno activo

    return frame

In [27]:
def generate_background_dark(t, flash_mode=False):
    """
    Genera un fondo hipnótico oscuro.
    Si flash_mode es True, aplica parpadeos y temblores de imagen.
    
    flash_mode: bool que activa efectos especiales de terror.
    """
    w, h = VIDEO_SIZE
    x = np.linspace(0, 4*np.pi, w)
    y = np.linspace(0, 4*np.pi, h)
    X, Y = np.meshgrid(x, y)

    # Onda base oscura
    wave = (np.sin(X * 2 + t*0.5) + np.sin(Y * 2 - t*1.5) + np.sin(X + Y + t)) * 0.33
    norm_wave = ((wave - wave.min()) / (wave.max() - wave.min()) * 255).astype(np.uint8)

    frame = np.zeros((h, w, 3), dtype=np.uint8)
    frame[..., 0] = (norm_wave // 3)    # Azul oscuro
    frame[..., 1] = (norm_wave // 8)    # Verde casi ausente
    frame[..., 2] = (norm_wave // 2) + 10  # Rojo profundo

    # --- Activar "modo terrorífico" ---
    if flash_mode:
        # 1. Parpadeo constante tipo luz defectuosa
        # flash_intensity = 1.5 + 0.5 * np.sin(10 * t)  # Oscila entre 1.0 y 2.0 veces el brillo
        flash_intensity = 1.5 + 0.2 * np.sin(10 * t)  # Oscila entre 1.0 y 2.0 veces el brillo
        frame = np.clip(frame * flash_intensity, 0, 255).astype(np.uint8)

        # 2. Temblor de imagen
        max_shift = 5  # máximo de 5 píxeles
        shift_x = int(np.random.uniform(-max_shift, max_shift))
        shift_y = int(np.random.uniform(-max_shift, max_shift))

        # Crear un frame desplazado
        M = np.float32([[1, 0, shift_x], [0, 1, shift_y]])
        frame = cv2.warpAffine(frame, M, (w, h), borderMode=cv2.BORDER_REFLECT)

    return frame

In [78]:
def generate_background_falling(t, zoom_speed=0.005):
    """
    Simula una caída infinita en un túnel oscuro tipo fractal.
    Usa coordenadas polares animadas.
    """
    w, h = VIDEO_SIZE
    x = np.linspace(-1, 1, w)
    y = np.linspace(-1, 1, h)
    X, Y = np.meshgrid(x, y)
    r = np.sqrt(X**2 + Y**2)
    theta = np.arctan2(Y, X)

    # Zoom animado para simular caída
    zoom = 1 + zoom_speed * t
    pattern = np.sin(10 * np.log(r * zoom + 1e-3) + t * 2 + theta * 5)

    # Normalizar patrón
    norm = ((pattern - pattern.min()) / (pattern.max() - pattern.min()) * 255).astype(np.uint8)

    frame = np.zeros((h, w, 3), dtype=np.uint8)
    # frame[..., 0] = norm // 2              # Azul oscuro
    # frame[..., 1] = norm // 10             # Verde tenue
    # frame[..., 2] = (255 - norm) // 3 + 5  # Rojo apagado

    # frame[..., 0] = norm // 2           # azul fuerte
    # frame[..., 1] = norm // 10          # verde bajo
    # frame[..., 2] = norm // 1.5         # rojo/púrpura

    frame[..., 0] = norm // 2           # azul fuerte
    frame[..., 1] = norm // 10          # verde bajo
    frame[..., 2] = norm // 10         # Todo AZUl


    # gray = norm
    # frame[..., 0] = gray
    # frame[..., 1] = gray
    # frame[..., 2] = gray


    return frame

In [4]:
def generate_background_falling(t, zoom_speed=0.005, colormap=cv2.COLORMAP_DEEPGREEN):
    """
    Simula una caída infinita con un mapa de color aplicable.
    """
    w, h = VIDEO_SIZE
    x = np.linspace(-1, 1, w)
    y = np.linspace(-1, 1, h)
    X, Y = np.meshgrid(x, y)
    r = np.sqrt(X**2 + Y**2)
    theta = np.arctan2(Y, X)

    zoom = 1 + zoom_speed * t
    pattern = np.sin(10 * np.log(r * zoom + 1e-3) + t * 2 + theta * 5)

    # Normalizar a escala de grises
    norm = ((pattern - pattern.min()) / (pattern.max() - pattern.min()) * 255).astype(np.uint8)

    # Aplicar mapa de colores
    frame = cv2.applyColorMap(norm, colormap)

    return frame

## Crea video

In [5]:
# Parámetros del video
# VIDEO_SIZE = (1280, 720)
# VIDEO_SIZE = (720,1280)
VIDEO_SIZE = (1920,1080)
# VIDEO_SIZE = (640,480)
FPS = 30
DURATION=30

from datetime import datetime
now = datetime.now()
formatted_time = now.strftime("%Y%m%d%H%M%S")
# print(formatted_time)

OUTPUT_FILE = "final_video_dark_"+formatted_time+".mp4"
FONT = cv2.FONT_HERSHEY_SIMPLEX
# flash_moments = [2.5, 5.7, 8.2]  # en qué segundos quieres los flashes

# Crear el video writer
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
video_writer = cv2.VideoWriter(OUTPUT_FILE, fourcc, FPS, VIDEO_SIZE)

# Generar todos los frames
total_frames = int(FPS * DURATION)
for frame_idx in range(total_frames):
    t = frame_idx / FPS  # tiempo en segundos
    flash_mode = (DURATION-t < 5)  # por ejemplo, que a partir de 5s empiece el caos
    # frame = generate_background_dark(t, flash_mode=flash_mode)
    frame = generate_background_falling(t)
    # Mostrar texto durante los primeros 5 segundos
    # if 0 < t < 5:
    # text = "El Carretero"
    # font_scale = 2
    # thickness = 3
    # (text_width, text_height), _ = cv2.getTextSize(text, FONT, font_scale, thickness)
    # text_x = (VIDEO_SIZE[0] - text_width) // 2
    # text_y = (VIDEO_SIZE[1] + text_height) // 2
    # cv2.putText(frame, text, (text_x, text_y), FONT, font_scale, (255, 255, 255), thickness, cv2.LINE_AA)

    frame = overlay_logo(frame)

    # Escribir frame en el video
    video_writer.write(frame)

# Liberar recursos
video_writer.release()
print("Video generado:", OUTPUT_FILE)

Video generado: final_video_dark_20250522022204.mp4


## Une audio y video

In [39]:
# ffmpeg -i final_video_dark_20250429055802.mp4 -i melody_tetrica_PulsodelaCriptaMejorada.wav -c:v copy -c:a aac -strict experimental ElCarreteroPilot.mp4

cmd = [
    'ffmpeg',
    '-i', 'final_video_dark_20250429055802.mp4',
    '-i', 'melody_tetrica_final20250429061321.wav',
    '-c:v', 'copy',
    '-c:a', 'aac',
    '-shortest',
    'ElCarreteroPilot_4.mp4'
]
subprocess.run(cmd)

ffmpeg version 5.1.6-0+deb12u1 Copyright (c) 2000-2024 the FFmpeg developers
  built with gcc 12 (Debian 12.2.0-14)
  configuration: --prefix=/usr --extra-version=0+deb12u1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librist --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtheora --enable-libtwolame --enable-libvidstab --enab

CompletedProcess(args=['ffmpeg', '-i', 'final_video_dark_20250429055802.mp4', '-i', 'melody_tetrica_final20250429061321.wav', '-c:v', 'copy', '-c:a', 'aac', '-shortest', 'ElCarreteroPilot_4.mp4'], returncode=0)

# Otro codigo para crear un video con voz

In [1]:
import cv2
import numpy as np
import pyttsx3
import threading
import time

# Historia sobrenatural
historia = """
Me llamo El Carretero, y esta historia me la contaron una noche en San Agustín del Valle.
Cuenta la gente que, cuando la neblina cae sin luna, una figura se aparece a los viajeros...
No camina. No respira. Solo observa. Y si tú la ves, ya es demasiado tarde.
"""

# Función para generar el fondo hipnótico
def generar_fondo_hipnotico(frame_num, width, height):
    img = np.zeros((height, width, 3), dtype=np.uint8)
    cx, cy = width // 2, height // 2
    for i in range(100):
        angle = (frame_num * 2 + i * 10) * np.pi / 180
        radius = 5 * i
        x = int(cx + radius * np.cos(angle))
        y = int(cy + radius * np.sin(angle))
        color = (i * 2 % 256, i * 5 % 256, 255 - i * 2 % 256)
        cv2.circle(img, (x, y), 10, color, -1)
    return img

# Función para narrar usando pyttsx3
def narrar(texto):
    engine = pyttsx3.init()
    engine.setProperty('rate', 150)
    engine.say(texto)
    engine.runAndWait()

# Función principal para crear el video
def crear_video_con_fondo_hipnotico(historia, output="historia.mp4", duracion=20, fps=30):
    width, height = 640, 480
    total_frames = duracion * fps
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output, fourcc, fps, (width, height))

    # Lanzar narración en paralelo
    narrador = threading.Thread(target=narrar, args=(historia,))
    narrador.start()

    # Generar video
    for i in range(total_frames):
        frame = generar_fondo_hipnotico(i, width, height)
        out.write(frame)
        time.sleep(1 / fps)

    out.release()
    narrador.join()
    print("Video generado:", output)

# Ejecutar
crear_video_con_fondo_hipnotico(historia)

Exception in thread Thread-4 (narrar):
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/site-packages/pyttsx3/__init__.py", line 20, in init
    eng = _activeEngines[driverName]
          ~~~~~~~~~~~~~~^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/weakref.py", line 136, in __getitem__
    o = self.data[key]()
        ~~~~~~~~~^^^^^
KeyError: None

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.11/threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "/usr/local/lib/python3.11/site-packages/ipykernel/ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "/usr/local/lib/python3.11/threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_343/2423747486.py", line 29, in narrar
  File "/usr/local/lib/python3.11/site-packages/pyttsx3/__init__.py", line 22, in init
    eng = Engine(driverName, debug)
    

Video generado: historia.mp4


In [6]:
import cv2
import numpy as np
import time

# Tamaño del video
VIDEO_SIZE = (1920, 1080)


In [ ]:
# def generate_background_dark(t, flash_mode=False):
#     """
#     Genera un fondo hipnótico oscuro.
#     Si flash_mode es True, aplica parpadeos y temblores de imagen.
#     """
#     w, h = VIDEO_SIZE
#     x = np.linspace(0, 4*np.pi, w)
#     y = np.linspace(0, 4*np.pi, h)
#     X, Y = np.meshgrid(x, y)

#     # Onda base oscura
#     wave = (np.sin(X * 2 + t*0.5) + np.sin(Y * 2 - t*1.5) + np.sin(X + Y + t)) * 0.33
#     norm_wave = ((wave - wave.min()) / (wave.max() - wave.min()) * 255).astype(np.uint8)

#     frame = np.zeros((h, w, 3), dtype=np.uint8)
#     frame[..., 0] = (norm_wave // 3)         # Azul oscuro
#     frame[..., 1] = (norm_wave // 8)         # Verde tenue
#     frame[..., 2] = (norm_wave // 2) + 10    # Rojo profundo

#     if flash_mode:
#         # Parpadeo
#         flash_intensity = 1.5 + 0.2 * np.sin(10 * t)
#         frame = np.clip(frame * flash_intensity, 0, 255).astype(np.uint8)

#         # Temblor de imagen
#         max_shift = 5
#         shift_x = int(np.random.uniform(-max_shift, max_shift))
#         shift_y = int(np.random.uniform(-max_shift, max_shift))

#         M = np.float32([[1, 0, shift_x], [0, 1, shift_y]])
#         frame = cv2.warpAffine(frame, M, (w, h), borderMode=cv2.BORDER_REFLECT)

#     return frame



# # Crear video con ese fondo
# def crear_video_fondo_dark(output="fondo_oscuro.mp4", duracion=30, fps=30, flash_mode=False):
#     total_frames = duracion * fps
#     width, height = VIDEO_SIZE
#     fourcc = cv2.VideoWriter_fourcc(*'mp4v')
#     out = cv2.VideoWriter(output, fourcc, fps, (width, height))

#     for i in range(total_frames):
#         t = i / fps
#         frame = generate_background_dark(t, flash_mode=flash_mode)
#         out.write(frame)

#     out.release()
#     print(f"Video generado: {output}")

# Ejecutar
# crear_video_fondo_dark(flash_mode=True)




Video generado: fondo_oscuro.mp4


In [28]:
def generate_background_falling(t, zoom_speed=0.005):
    """
    Simula una caída infinita en un túnel oscuro tipo fractal.
    Usa coordenadas polares animadas.
    """
    w, h = VIDEO_SIZE
    x = np.linspace(-1, 1, w)
    y = np.linspace(-1, 1, h)
    X, Y = np.meshgrid(x, y)
    r = np.sqrt(X**2 + Y**2)
    theta = np.arctan2(Y, X)

    # Zoom animado para simular caída
    zoom = 1 + zoom_speed * t
    pattern = np.sin(10 * np.log(r * zoom + 1e-3) + t * 2 + theta * 5)

    # Normalizar patrón
    norm = ((pattern - pattern.min()) / (pattern.max() - pattern.min()) * 255).astype(np.uint8)

    frame = np.zeros((h, w, 3), dtype=np.uint8)
    frame[..., 0] = norm // 2              # Azul oscuro
    frame[..., 1] = norm // 10             # Verde tenue
    frame[..., 2] = (255 - norm) // 3 + 5  # Rojo apagado

    return frame

def crear_video_fondo_falling(output="fondo_caida_infinita2.mp4", duracion=5, fps=30):
    total_frames = duracion * fps
    width, height = VIDEO_SIZE
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output, fourcc, fps, (width, height))

    for i in range(total_frames):
        t = i / fps
        frame = generate_background_falling(t)
        out.write(frame)

    out.release()
    print(f"Video generado: {output}")

crear_video_fondo_falling()


Video generado: fondo_caida_infinita2.mp4


In [9]:
def generate_background_falling(t, zoom_speed=0.05, colormap=cv2.COLORMAP_BONE):
    """
    Simula una caída infinita con un mapa de color aplicable.
    """
    w, h = VIDEO_SIZE
    x = np.linspace(-1, 1, w)
    y = np.linspace(-1, 1, h)
    X, Y = np.meshgrid(x, y)
    r = np.sqrt(X**2 + Y**2)
    theta = np.arctan2(Y, X)

    zoom = 1 + zoom_speed * t
    pattern = np.sin(10 * np.log(r * zoom + 1e-3) + t * 2 + theta * 5)

    # Normalizar a escala de grises
    norm = ((pattern - pattern.min()) / (pattern.max() - pattern.min()) * 255).astype(np.uint8)

    # Aplicar mapa de colores
    frame = cv2.applyColorMap(norm, colormap)

    return frame

def crear_video_fondo_falling(output="fondo_caida_infinita_COLORMAP_BONE.mp4", duracion=5, fps=30):
    total_frames = duracion * fps
    width, height = VIDEO_SIZE
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output, fourcc, fps, (width, height))

    for i in range(total_frames):
        t = i / fps
        frame = generate_background_falling(t)
        out.write(frame)

    out.release()
    print(f"Video generado: {output}")

crear_video_fondo_falling()

Video generado: fondo_caida_infinita_COLORMAP_BONE.mp4


# Trabajando con la aparición de las palabras



In [ ]:
def overlay_fading_text(frame, text, t, start_time=0, duration=5, position="center", font_scale=1.2, thickness=2):
    """
    Superpone texto en el frame que se desvanece con el tiempo.
    - t: tiempo actual en segundos
    - start_time: momento en que aparece el texto
    - duration: cuánto dura antes de desaparecer
    """

    fade = max(0, min(1, 1 - (t - start_time) / duration))  # 1 → 0
    if fade == 0:
        return frame

    overlay = frame.copy()
    h, w = frame.shape[:2]

    font = cv2.FONT_HERSHEY_SIMPLEX
    text_size, _ = cv2.getTextSize(text, font, font_scale, thickness)
    text_w, text_h = text_size

    if position == "center":
        pos = (w // 2 - text_w // 2, h // 2 + text_h // 2)
    else:
        pos = position  # (x, y)

    # Color blanco con opacidad
    color = (255, 255, 255)
    alpha = fade

    # --- Sombra (negra desplazada) ---
    shadow_offset = 2
    shadow_pos = (pos[0] + shadow_offset, pos[1] + shadow_offset)
    cv2.putText(overlay, text, shadow_pos, font, font_scale, (0, 0, 0), thickness + 2, cv2.LINE_AA)

    # --- Texto principal ---
    cv2.putText(overlay, text, pos, font, font_scale, color, thickness, cv2.LINE_AA)

    # Mezcla con el fondo
    cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0, frame)

    return frame

In [73]:
def overlay_fading_text_unicode(frame, text, t, start_time=0, duration=5, position="center",
                                font_path="DejaVuSans.ttf", max_font_size=48, max_width_ratio=0.9):
    """
    Superpone texto UTF-8 con fade-out.
    - Usa Pillow para soportar acentos y caracteres especiales.
    - Ajusta automáticamente tamaño y división en líneas.
    """

    fade = max(0, min(1, 1 - (t - start_time) / duration))
    if fade == 0:
        return frame

    h, w = frame.shape[:2]
    image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(image)

    # Preparar fuente
    font_size = max_font_size
    font = ImageFont.truetype(font_path, font_size)

    # Dividir texto si excede el ancho máximo
    words = text.split()
    lines = []
    current_line = words[0]

    for word in words[1:]:
        test_line = current_line + ' ' + word
        if draw.textlength(test_line, font=font) > w * max_width_ratio:
            lines.append(current_line)
            current_line = word
        else:
            current_line = test_line
    lines.append(current_line)

    # Reducir fuente si las líneas siguen siendo muy anchas
    while True:
        too_wide = any(draw.textlength(line, font=font) > w * max_width_ratio for line in lines)
        if not too_wide:
            break
        font_size = int(font_size * 0.9)
        font = ImageFont.truetype(font_path, font_size)

    # Altura de línea usando bbox (compatible con Pillow moderno)
    bbox = font.getbbox("Ay")  # bounding box del texto
    line_height = (bbox[3] - bbox[1]) + 6
    total_height = len(lines) * line_height

    # Calcular posición vertical
    if position == "center":
        y0 = h // 2 - total_height // 2
    else:
        _, y0 = position

    # Dibujar cada línea con sombra y transparencia
    for i, line in enumerate(lines):
        text_w = draw.textlength(line, font=font)
        if position == "center":
            x = (w - text_w) // 2
        else:
            x, _ = position
        y = y0 + i * line_height

        # Sombra
        draw.text((x + 2, y + 2), line, font=font, fill=(0, 0, 0, int(255 * fade)))
        # Texto principal
        draw.text((x, y), line, font=font, fill=(255, 255, 255, int(255 * fade)))

    # Convertir de vuelta a OpenCV
    frame_result = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
    return frame_result

Las letras desaparecen por detras

In [ ]:
def overlay_letterwise_fadein_fadeout_text(frame, text, t, start_time=0, duration=5,position="center", font_path="DejaVuSans.ttf",max_font_size=48, max_width_ratio=0.9):
    """
    Superpone texto con aparición tipo máquina de escribir + desaparición letra por letra.
    """
    h, w = frame.shape[:2]
    image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(image)

    # Preparar fuente
    font_size = max_font_size
    font = ImageFont.truetype(font_path, font_size)

    # Dividir texto en líneas según ancho máximo
    words = text.split()
    lines = []
    current_line = words[0]
    for word in words[1:]:
        test_line = current_line + ' ' + word
        if draw.textlength(test_line, font=font) > w * max_width_ratio:
            lines.append(current_line)
            current_line = word
        else:
            current_line = test_line
    lines.append(current_line)

    # Reducir tamaño de fuente si sigue siendo muy ancha
    while any(draw.textlength(line, font=font) > w * max_width_ratio for line in lines):
        font_size = int(font_size * 0.9)
        font = ImageFont.truetype(font_path, font_size)

    # Calcular posición vertical
    bbox = font.getbbox("Ay")
    line_height = (bbox[3] - bbox[1]) + 6
    total_height = len(lines) * line_height
    y0 = h // 2 - total_height // 2 if position == "center" else position[1]

    # Duraciones
    elapsed = t - start_time
    fade_in_time = duration * 0.3
    fade_out_time = duration * 0.3
    mid_time = duration - fade_out_time

    # Cuántos caracteres mostrar (máquina de escribir + desaparición)
    total_chars = sum(len(line) for line in lines)

    if elapsed <= 0 or elapsed >= duration:
        return frame  # nada que mostrar

    if elapsed < fade_in_time:
        ratio = elapsed / fade_in_time
        visible_chars = int(total_chars * ratio)
    elif elapsed > mid_time:
        ratio = 1 - ((elapsed - mid_time) / fade_out_time)
        visible_chars = int(total_chars * ratio)
    else:
        visible_chars = total_chars

    # Dibujar letra por letra
    char_count = 0
    for i, line in enumerate(lines):
        y = y0 + i * line_height
        text_w = draw.textlength(line, font=font)
        x0 = (w - text_w) // 2 if position == "center" else position[0]
        x = x0

        for ch in line:
            if char_count < visible_chars:
                # Sombra
                draw.text((x + 2, y + 2), ch, font=font, fill=(0, 0, 0))
                # Texto principal
                draw.text((x, y), ch, font=font, fill=(255, 255, 255))
            x += draw.textlength(ch, font=font)
            char_count += 1

    return cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)

In [85]:
def overlay_letterwise_fadein_fadeout_text(frame, text, t, start_time=0, duration=5,
                                           position="center", font_path="DejaVuSans.ttf",
                                           max_font_size=48, max_width_ratio=0.9):
    """
    Superpone texto con aparición tipo máquina de escribir y desaparición desde el inicio.
    """
    h, w = frame.shape[:2]
    image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(image)

    # Preparar fuente
    font_size = max_font_size
    font = ImageFont.truetype(font_path, font_size)

    # Dividir texto en líneas según ancho máximo
    words = text.split()
    lines = []
    current_line = words[0]
    for word in words[1:]:
        test_line = current_line + ' ' + word
        if draw.textlength(test_line, font=font) > w * max_width_ratio:
            lines.append(current_line)
            current_line = word
        else:
            current_line = test_line
    lines.append(current_line)

    # Ajustar fuente si hay líneas demasiado anchas
    while any(draw.textlength(line, font=font) > w * max_width_ratio for line in lines):
        font_size = int(font_size * 0.9)
        font = ImageFont.truetype(font_path, font_size)

    # Posición vertical
    bbox = font.getbbox("Ay")
    line_height = (bbox[3] - bbox[1]) + 6
    total_height = len(lines) * line_height
    y0 = h // 2 - total_height // 2 if position == "center" else position[1]

    # Duraciones
    elapsed = t - start_time
    fade_in_time = duration * 0.3
    fade_out_time = duration * 0.3
    mid_time = duration - fade_out_time
    total_chars = sum(len(line) for line in lines)

    # Determinar cuántos caracteres se deben mostrar
    if elapsed <= 0 or elapsed >= duration:
        return frame

    if elapsed < fade_in_time:
        ratio = elapsed / fade_in_time
        visible_chars = int(total_chars * ratio)
    elif elapsed > mid_time:
        ratio = 1 - ((elapsed - mid_time) / fade_out_time)
        visible_chars = int(total_chars * ratio)
    else:
        visible_chars = total_chars

    # Dibujar caracteres visibles (siempre desde el inicio del texto)
    char_count = 0
    for i, line in enumerate(lines):
        y = y0 + i * line_height
        text_w = draw.textlength(line, font=font)
        x0 = (w - text_w) // 2 if position == "center" else position[0]
        x = x0

        for ch in line:
            if char_count < visible_chars:
                draw.text((x + 2, y + 2), ch, font=font, fill=(0, 0, 0))          # sombra
                draw.text((x, y), ch, font=font, fill=(255, 255, 255))            # letra
            x += draw.textlength(ch, font=font)
            char_count += 1

    return cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)

In [6]:
def overlay_letterwise_fadein_fadeout_text(frame, text, t, start_time=0, duration=5,
                                           position="center", font_path="DejaVuSans.ttf",
                                           max_font_size=48, max_width_ratio=0.9):
    h, w = frame.shape[:2]
    image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(image)

    # Fuente
    font_size = max_font_size
    font = ImageFont.truetype(font_path, font_size)

    # Separar texto en líneas
    words = text.split()
    lines = []
    current_line = words[0]
    for word in words[1:]:
        test_line = current_line + ' ' + word
        if draw.textlength(test_line, font=font) > w * max_width_ratio:
            lines.append(current_line)
            current_line = word
        else:
            current_line = test_line
    lines.append(current_line)

    # Reducir fuente si alguna línea es muy ancha
    while any(draw.textlength(line, font=font) > w * max_width_ratio for line in lines):
        font_size = int(font_size * 0.9)
        font = ImageFont.truetype(font_path, font_size)

    # Posición vertical
    bbox = font.getbbox("Ay")
    line_height = (bbox[3] - bbox[1]) + 6
    total_height = len(lines) * line_height
    y0 = h // 2 - total_height // 2 if position == "center" else position[1]

    # Duración de cada fase
    elapsed = t - start_time
    fade_in_time = duration * 0.3
    fade_out_time = duration * 0.2
    mid_time = duration - fade_out_time
    total_chars = sum(len(line) for line in lines)

    if elapsed <= 0 or elapsed >= duration:
        return frame

    # Calcular qué letras mostrar
    if elapsed < fade_in_time:
        ratio = elapsed / fade_in_time
        start_char = 0
        end_char = int(total_chars * ratio)
    elif elapsed > mid_time:
        ratio = (elapsed - mid_time) / fade_out_time
        start_char = int(total_chars * ratio)
        end_char = total_chars
    else:
        start_char = 0
        end_char = total_chars

    # Dibujar letras visibles
    char_index = 0
    for i, line in enumerate(lines):
        y = y0 + i * line_height
        text_w = draw.textlength(line, font=font)
        x0 = (w - text_w) // 2 if position == "center" else position[0]
        x = x0

        for ch in line:
            if start_char <= char_index < end_char:
                draw.text((x + 2, y + 2), ch, font=font, fill=(0, 0, 0))       # sombra
                draw.text((x, y), ch, font=font, fill=(255, 255, 255))         # texto
            x += draw.textlength(ch, font=font)
            char_index += 1

    return cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)

In [5]:
class DuracionFrase():
    def __init__(self, frase, duracion,acumulado):
        self.frase = frase
        self.duracion = duracion
        self.acumulado= acumulado

text="Hola, yo soy El Carretero, y en mis multiples viajes he visto muchas cosas, y he escuchado infinidad de historias... hoy te contaré una a la que llamo: La niebla todavía habla."

duracionFrase=[]
textmodif=text.replace("...",",").replace(":",",").replace(".","")
print(textmodif)
acum=0
frasescortas=textmodif.split(",")
for frase in frasescortas:
	tamFrase=len(frase.split(" "))
	acum+=tamFrase
	duracionFrase.append(DuracionFrase(frase,len(frase.split(" "))/9,acum))
	print(frase,tamFrase,acum)

# # duracionxfrase = np.array([frasescortas,duraciones])
# for df in duracionFrase:
# 	print(df.frase,df.duracion,df.acumulado)

Hola, yo soy El Carretero, y en mis multiples viajes he visto muchas cosas, y he escuchado infinidad de historias, hoy te contaré una a la que llamo, La niebla todavía habla
Hola 1 1
 yo soy El Carretero 5 6
 y en mis multiples viajes he visto muchas cosas 10 16
 y he escuchado infinidad de historias 7 23
 hoy te contaré una a la que llamo 9 32
 La niebla todavía habla 5 37


In [8]:
text="Hola, yo soy El Carretero, y en mis multiples viajes he visto muchas cosas, y he escuchado infinidad de historias... hoy te contaré una a la que llamo: La niebla todavía habla."
duracionTexto=9
totalPalabras=len(text.split(" "))
tiempoxPalabra=duracionTexto/totalPalabras
print(duracionTexto,totalPalabras,tiempoxPalabra)

textmodif=text.replace("...",",").replace(":",",").replace(".","")
frasescortas=textmodif.split(",")
print(frasescortas,len(frasescortas))
duraciones=[]

for frase in frasescortas:
	palabras=len(frase.split(" "))
	duraciones.append(palabras*tiempoxPalabra)

print(frasescortas)
print(duraciones)

9 32 0.28125
['Hola', ' yo soy El Carretero', ' y en mis multiples viajes he visto muchas cosas', ' y he escuchado infinidad de historias', ' hoy te contaré una a la que llamo', ' La niebla todavía habla'] 6
['Hola', ' yo soy El Carretero', ' y en mis multiples viajes he visto muchas cosas', ' y he escuchado infinidad de historias', ' hoy te contaré una a la que llamo', ' La niebla todavía habla']
[0.28125, 1.40625, 2.8125, 1.96875, 2.53125, 1.40625]


In [11]:
vector = np.array([1, 2, 3])
norm = np.linalg.norm(vector)
normalized_vector = vector / norm

print(normalized_vector)
normalized_vector=normalized_vector.round(decimals=1)
print(normalized_vector)

[0.26726124 0.53452248 0.80178373]
[0.3 0.5 0.8]


In [10]:
# Parámetros del video
# VIDEO_SIZE = (1280, 720)
# VIDEO_SIZE = (720,1280)
VIDEO_SIZE = (1920,1080)
# VIDEO_SIZE = (640,480)
FPS = 30
DURATION=60

from datetime import datetime
now = datetime.now()
formatted_time = now.strftime("%Y%m%d%H%M%S")
# print(formatted_time)

OUTPUT_FILE = "final_video_dark_"+formatted_time+".mp4"
FONT = cv2.FONT_HERSHEY_SIMPLEX
# flash_moments = [2.5, 5.7, 8.2]  # en qué segundos quieres los flashes

# Crear el video writer
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
video_writer = cv2.VideoWriter(OUTPUT_FILE, fourcc, FPS, VIDEO_SIZE)

contadorFrasesCortas=1
# Generar todos los frames
total_frames = int(FPS * DURATION)
for frame_idx in range(total_frames):
	t = frame_idx / FPS  # tiempo en segundos
	flash_mode = (DURATION-t < 5)  # por ejemplo, que a partir de 5s empiece el caos
	# frame = generate_background_dark(t, flash_mode=flash_mode)
	frame = generate_background_falling(t)
	# Mostrar texto durante los primeros 5 segundos
	# if 0 < t < 5:
	# text = "El Carretero"
	
	# if contadorFrasesCortas >= duracionFrase[contadorFrasesCortas].acumulado:
		

	# font_scale = 2
	# thickness = 3
	# (text_width, text_height), _ = cv2.getTextSize(text, FONT, font_scale, thickness)
	# text_x = (VIDEO_SIZE[0] - text_width) // 2
	# text_y = (VIDEO_SIZE[1] + text_height) // 2
	# cv2.putText(frame, text, (text_x, text_y), FONT, font_scale, (255, 255, 255), thickness, cv2.LINE_AA)

	# frame = overlay_fading_text(frame, text, t,start_time=2)
	# frame = overlay_fading_text_unicode(frame, text, t,start_time=2)
	# frame = overlay_letterwise_fadein_fadeout_text(frame, text, t,start_time=2)
	frame = overlay_letterwise_fadein_fadeout_text(frame, text, t,start_time=0,duration=5)

	# Escribir frame en el video
	video_writer.write(frame)

# Liberar recursos
video_writer.release()
print("Video generado:", OUTPUT_FILE)

Video generado: final_video_dark_20250522030243.mp4


In [ ]:
frasesCortas=text.replace("...",",").replace(":",",").split(",")
print(len(frasesCortas))


6


In [94]:
text="Hola, yo soy El Carretero, y en mis multiples viajes he visto muchas cosas, y he escuchado infinidad de historias... hoy te contaré una a la que llamo: La niebla todavía habla."
# text="Hola yo,"
print(len(set(text)))

31
